In [2]:

import pickle
import pandas as pd
from ete3 import NCBITaxa
from tqdm import tqdm
import numpy as np

In [10]:
# /scratch/suyuelyu/deimm/data/oma/metadata_all_w_seq.parquet
all_meta = pd.read_parquet('/scratch/suyuelyu/deimm/data/oma/metadata_all.parquet')
all_meta.head()

,protein,taxid,lineage,og
0,ARCGX00001,1579367,"[131567, 2157, 93506, 1579367]",1220080
1,ARCGX00005,1579367,"[131567, 2157, 93506, 1579367]",962367
2,ARCGX00007,1579367,"[131567, 2157, 93506, 1579367]",775166
3,ARCGX00008,1579367,"[131567, 2157, 93506, 1579367]",1189658
4,ARCGX00009,1579367,"[131567, 2157, 93506, 1579367]",963202


In [11]:
# get full lineage at all standard ranks
ncbi = NCBITaxa()
std_ranks = ['domain', 'phylum', 'class', 'order', 'family', 'genus', 'species']
# get std rank for uniq taxid
def get_std_rankid(taxid):
    lineage = ncbi.get_lineage(taxid)
    ranks = ncbi.get_rank(lineage)
    std_ranks_dict = {ranks[t]: t for t in lineage if ranks[t] in std_ranks}
    return std_ranks_dict
uniq_taxids = all_meta['taxid'].unique()
taxid_to_std_ranks = {taxid: get_std_rankid(taxid) for taxid in uniq_taxids}

/scratch/suyuelyu/deimm/deimm/.venv/lib/python3.11/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 2670412 was translated into 2666348
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/scratch/suyuelyu/deimm/deimm/.venv/lib/python3.11/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 2052317 was translated into 3107663
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/scratch/suyuelyu/deimm/deimm/.venv/lib/python3.11/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 1081105 was translated into 1649241
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/scratch/suyuelyu/deimm/deimm/.venv/lib/python3.11/site-packages/ete3/ncbi_taxonomy/ncbiquery.py:243: UserWarning: taxid 1064592 was translated into 27288
  warnings.warn("taxid %s was translated into %s" %(taxid, merged_conversion[taxid]))
/scratch/suyuelyu/deimm/deimm/

In [12]:
# save taxid_to_std_ranks as pickle
with open('/scratch/suyuelyu/deimm/data/oma/taxid_to_std_ranks.pkl', 'wb') as f:
    pickle.dump(taxid_to_std_ranks, f)

In [16]:
# check if anything is missing species rank
missing_species = [taxid for taxid, ranks in taxid_to_std_ranks.items() if 'species' not in ranks]
print(f"Taxids missing species rank: {missing_species}")

Taxids missing species rank: []


In [49]:
for rank in tqdm(std_ranks):
    all_meta[rank] = all_meta['taxid'].apply(lambda x: taxid_to_std_ranks[int(x)].get(rank, np.nan))
all_meta.head()

100%|██████████| 7/7 [00:35<00:00,  5.03s/it]


,protein,taxid,lineage,og,domain,phylum,class,order,family,genus,species
0,ARCGX00001,1579367,"[131567, 2157, 93506, 1579367]",1220080,2157,NaN,NaN,NaN,NaN,NaN,1579367
1,ARCGX00005,1579367,"[131567, 2157, 93506, 1579367]",962367,2157,NaN,NaN,NaN,NaN,NaN,1579367
2,ARCGX00007,1579367,"[131567, 2157, 93506, 1579367]",775166,2157,NaN,NaN,NaN,NaN,NaN,1579367
3,ARCGX00008,1579367,"[131567, 2157, 93506, 1579367]",1189658,2157,NaN,NaN,NaN,NaN,NaN,1579367
4,ARCGX00009,1579367,"[131567, 2157, 93506, 1579367]",963202,2157,NaN,NaN,NaN,NaN,NaN,1579367


In [ ]:
# $SCRATCH/mmseqs/bin/mmseqs easy-linclust /scratch/suyuelyu/deimm/data/oma/oma-seqs.fa /scratch/suyuelyu/deimm/data/oma/oma-seqs-linclust-50pident tmp --min-seq-id 0.5
# /scratch/suyuelyu/deimm/data/oma/oma-seqs-linclust-50pident_cluster.tsv
linclust_df = pd.read_csv('/scratch/suyuelyu/deimm/data/oma/oma-seqs-linclust-50pident_cluster.tsv', sep='\t', header=None, names=['rep_id', 'seq_id'])
# each cluster should have a cluster number starting from 0 to n clusters-1
n_clusters = linclust_df['rep_id'].nunique()

In [31]:
# add cluster id in descending order of cluster size
cluster_sizes = linclust_df['rep_id'].value_counts().sort_values(ascending=False)
cluster_id_mapping = {old_id: new_id for new_id, old_id in enumerate(cluster_sizes.index)}
linclust_df['cluster_id'] = linclust_df['rep_id'].map(cluster_id_mapping)
linclust_df.head()

,rep_id,seq_id,cluster_id
0,BRAM501876,BRAM501876,2269813
1,BRAM501876,RUMCH02411,2269813
2,BRAM501877,BRAM501877,716842
3,BRAM501877,BRAHW02137,716842
4,BRAM501877,BRAIP02462,716842


In [34]:
print(f"Number of clusters: {n_clusters}")

Number of clusters: 9718193


In [36]:
linclust_df['cluster_id'].value_counts().head()

cluster_id
0    1238
1    1125
2    1051
3    1049
4     964
Name: count, dtype: int64

In [37]:
all_meta.shape, linclust_df.shape

((15199426, 11), (22083295, 3))

In [ ]:
seq_id_to_cluster_id = dict(zip(linclust_df['seq_id'], linclust_df['cluster_id']))

In [51]:
all_meta["cluster_id"] = all_meta["protein"].map(seq_id_to_cluster_id)
all_meta.head()

,protein,taxid,lineage,og,domain,phylum,class,order,family,genus,species,cluster_id
0,ARCGX00001,1579367,"[131567, 2157, 93506, 1579367]",1220080,2157,NaN,NaN,NaN,NaN,NaN,1579367,6740521
1,ARCGX00005,1579367,"[131567, 2157, 93506, 1579367]",962367,2157,NaN,NaN,NaN,NaN,NaN,1579367,6740532
2,ARCGX00007,1579367,"[131567, 2157, 93506, 1579367]",775166,2157,NaN,NaN,NaN,NaN,NaN,1579367,6740530
3,ARCGX00008,1579367,"[131567, 2157, 93506, 1579367]",1189658,2157,NaN,NaN,NaN,NaN,NaN,1579367,6740529
4,ARCGX00009,1579367,"[131567, 2157, 93506, 1579367]",963202,2157,NaN,NaN,NaN,NaN,NaN,1579367,6740528


In [52]:
# remove any rows with NA in any of the std ranks then how many species and rows left?
all_meta_no_na = all_meta.dropna(subset=std_ranks, how='any')
print(f"Number of rows with no NA in std ranks: {len(all_meta_no_na)}")
print(
    f"Number of unique species with no NA in std ranks: {all_meta_no_na['species'].nunique()}"
)
# n clusters with no NA in std ranks?
print(f"Number of clusters with no NA in std ranks: {all_meta_no_na['cluster_id'].nunique()}"
)

Number of rows with no NA in std ranks: 13989546
Number of unique species with no NA in std ranks: 1908
Number of clusters with no NA in std ranks: 5276030


In [53]:
# save the all_meta with cluster_id to parquet
all_meta.to_parquet('/scratch/suyuelyu/deimm/data/oma/metadata_all_with_cluster.parquet', index=False)

In [71]:
# any og with multiple clusters?
og_cluster_counts = all_meta_no_na.groupby("og")["cluster_id"].nunique()
# any clusters with multiple ogs?
cluster_og_counts = all_meta_no_na.groupby("cluster_id")["og"].nunique()
print(f"Number of OGs with multiple clusters: {(og_cluster_counts > 1).sum()}")
print(f"Number of clusters with multiple OGs: {(cluster_og_counts > 1).sum()}")

Number of OGs with multiple clusters: 928942
Number of clusters with multiple OGs: 621655


In [67]:
# for each og get unique std ranks it covers and number of clusters
og_std_ranks = all_meta_no_na.groupby("og")[std_ranks].nunique()
og_cluster_counts = all_meta_no_na.groupby("og")["cluster_id"].nunique()
# number of samples per og?
og_nsequences = all_meta_no_na.groupby("og")["protein"].nunique()
og_summary = pd.concat([og_std_ranks, og_cluster_counts.rename("cluster_count"), og_nsequences.rename("n_sequences")], axis=1)

In [ ]:
# sort decending by domain, then phylum, then class, then order, then family, then genus, then species,
og_summary_sorted = og_summary.sort_values(by=std_ranks, ascending=False)

In [69]:
# drop anything with single species coverage
og_summary_sorted_multi_species = og_summary_sorted[og_summary_sorted['species'] > 1]
og_summary_sorted_multi_species

,domain,phylum,class,order,family,genus,species,cluster_count,n_sequences
og,,,,,,,,,
1250885,3,82,182,378,654,1106,1747,198,2267
1146506,3,82,178,350,599,1015,1629,361,2138
953976,3,81,186,364,599,937,1347,133,1536
1238393,3,81,179,365,636,1060,1638,569,2065
1212410,3,81,175,376,661,1081,1730,299,2250
...,...,...,...,...,...,...,...,...,...
99965,1,1,1,1,1,1,2,1,3
99980,1,1,1,1,1,1,2,1,3
9999,1,1,1,1,1,1,2,2,2


In [70]:
for rank in std_ranks:
    # print n og with multi unique values for this rank
    n_og_multi_rank = (og_summary_sorted_multi_species[rank] > 1).sum()
    print(f"Number of OGs with multiple unique {rank}: {n_og_multi_rank}")
    n_og_single_rank = (og_summary_sorted_multi_species[rank] == 1).sum()
    print(f"Number of OGs with single unique {rank}: {n_og_single_rank}")

Number of OGs with multiple unique domain: 30555
Number of OGs with single unique domain: 1011761
Number of OGs with multiple unique phylum: 258647
Number of OGs with single unique phylum: 783669
Number of OGs with multiple unique class: 417618
Number of OGs with single unique class: 624698
Number of OGs with multiple unique order: 655774
Number of OGs with single unique order: 386542
Number of OGs with multiple unique family: 741709
Number of OGs with single unique family: 300607
Number of OGs with multiple unique genus: 868690
Number of OGs with single unique genus: 173626
Number of OGs with multiple unique species: 1042316
Number of OGs with single unique species: 0


In [129]:
val_test_og = []
seed = 351
n_per_group = 100
for i, rank in enumerate(std_ranks):
    # pick val test group with single R vs multi R
    og_single_rank = og_summary_sorted_multi_species[og_summary_sorted_multi_species[rank] == 1]
    if i > 0:
        # need to make sure multi R is within same R-1 group
        prev_rank = std_ranks[i-1]
        og_multi_rank = og_summary_sorted_multi_species[(og_summary_sorted_multi_species[rank] > 1) & (og_summary_sorted_multi_species[prev_rank] == 1)]
    else:
        og_multi_rank = og_summary_sorted_multi_species[og_summary_sorted_multi_species[rank] > 1]
    # rank both by n next rank
    if i < len(std_ranks) - 1:
        next_rank = std_ranks[i+1]
        og_single_rank = og_single_rank.sort_values(by=next_rank, ascending=False)
        og_multi_rank = og_multi_rank.sort_values(by=next_rank, ascending=False)
    else:
        og_single_rank = og_single_rank.sort_values(by=rank, ascending=False)
        og_multi_rank = og_multi_rank.sort_values(by=rank, ascending=False)

    # take top 1000 from each group to sample from
    og_single_rank = og_single_rank.head(500)
    og_multi_rank = og_multi_rank.head(500)
    
    # randomly sample 3 og from each group
    if len(og_single_rank) >= n_per_group:
        og_single_rank_sample = og_single_rank.sample(
            n=n_per_group, random_state=seed
        ).index.tolist()
        seed += 10
        val_test_og.append((rank, "single", og_single_rank_sample))
    if len(og_multi_rank) >= n_per_group:
        og_multi_rank_sample = og_multi_rank.sample(
            n=n_per_group, random_state=seed
        ).index.tolist()
        seed += 10
        val_test_og.append((rank, 'multi', og_multi_rank_sample))
val_test_og_df = pd.DataFrame(val_test_og, columns=['rank', 'group', 'ogs'])
val_test_og_flat = [og for rank, group, ogs in val_test_og for og in ogs]
meta_val_test = all_meta_no_na[all_meta_no_na["og"].isin(val_test_og_flat)]
meta_train = all_meta_no_na[~all_meta_no_na["og"].isin(val_test_og_flat)]
print(f"Number of val test sequences: {len(meta_val_test)}")
print(f"Number of val test OGs: {meta_val_test['og'].nunique()}")
print(f"Number of train sequences: {len(meta_train)}")
print(f"Number of train OGs: {meta_train['og'].nunique()}")
# cluster ids in val test set
val_test_cluster_ids = meta_val_test["cluster_id"].unique()
# how many seq i meta train have cluster ids in val test set?
n_train_with_val_test_cluster = (
    meta_train["cluster_id"].isin(val_test_cluster_ids).sum()
)
print(
    f"Number of training sequences with cluster ids in val test set: {n_train_with_val_test_cluster}"
)
# remove any training sequences with cluster ids in val test set
meta_train_no_cluster_leak = meta_train[
    ~meta_train["cluster_id"].isin(val_test_cluster_ids)
]
print(
    f"Number of training sequences after removing cluster leak: {len(meta_train_no_cluster_leak)}"
)
print(
    f"Number of training OGs after removing cluster leak: {meta_train_no_cluster_leak['og'].nunique()}"
)

Number of val test sequences: 286244
Number of val test OGs: 1255
Number of train sequences: 13703302
Number of train OGs: 1156312
Number of training sequences with cluster ids in val test set: 45548
Number of training sequences after removing cluster leak: 13657754
Number of training OGs after removing cluster leak: 1155981


In [130]:
val_test_og_df = pd.DataFrame(val_test_og, columns=["rank", "group", "ogs"])
val_test_og_df

,rank,group,ogs
0,domain,single,"[770260, 930189, 1240134, 961038, 1238252, 122..."
1,domain,multi,"[1248695, 921722, 1236080, 1217811, 711362, 12..."
2,phylum,single,"[1239313, 343149, 341795, 1244090, 1215031, 81..."
3,phylum,multi,"[699505, 1203908, 960958, 960947, 1235292, 691..."
4,class,single,"[923689, 546708, 618759, 673331, 547234, 54583..."
5,class,multi,"[349252, 1238424, 941963, 1156219, 967819, 115..."
6,order,single,"[815076, 816970, 831377, 868566, 853980, 34230..."
7,order,multi,"[1183778, 413092, 364521, 415668, 355402, 9799..."
8,family,single,"[943028, 261321, 858258, 831030, 1171925, 8535..."
9,family,multi,"[151899, 3782, 12994, 1437, 973362, 2428, 1512..."


In [131]:
# get size of each og in val test set
val_test_og_to_og_size = all_meta_no_na.groupby("og")["protein"].nunique().to_dict()
val_test_og_df['og_size'] = val_test_og_df['ogs'].apply(lambda ogs: [val_test_og_to_og_size[og] for og in ogs])

In [132]:
val_test_og_df['min_og_size'] = val_test_og_df['og_size'].apply(lambda sizes: min(sizes))
val_test_og_df['max_og_size'] = val_test_og_df['og_size'].apply(lambda sizes: max(sizes))
val_test_og_df['mean_og_size'] = val_test_og_df['og_size'].apply(lambda sizes: np.mean(sizes))
val_test_og_df

,rank,group,ogs,og_size,min_og_size,max_og_size,mean_og_size
0,domain,single,"[770260, 930189, 1240134, 961038, 1238252, 122...","[493, 455, 502, 518, 1061, 107, 490, 1353, 480...",107,1353,572.30
1,domain,multi,"[1248695, 921722, 1236080, 1217811, 711362, 12...","[2118, 1620, 1988, 1239, 1477, 1433, 1662, 158...",460,2254,1337.25
2,phylum,single,"[1239313, 343149, 341795, 1244090, 1215031, 81...","[120, 142, 135, 135, 85, 42, 103, 109, 122, 14...",17,147,105.09
3,phylum,multi,"[699505, 1203908, 960958, 960947, 1235292, 691...","[524, 504, 515, 525, 476, 443, 535, 475, 567, ...",404,1489,544.40
4,class,single,"[923689, 546708, 618759, 673331, 547234, 54583...","[43, 59, 46, 54, 58, 47, 52, 58, 59, 60, 55, 5...",40,66,55.09
5,class,multi,"[349252, 1238424, 941963, 1156219, 967819, 115...","[136, 134, 133, 141, 133, 139, 139, 133, 133, ...",121,144,134.13
6,order,single,"[815076, 816970, 831377, 868566, 853980, 34230...","[12, 13, 18, 15, 16, 14, 15, 14, 13, 15, 18, 1...",10,136,18.49
7,order,multi,"[1183778, 413092, 364521, 415668, 355402, 9799...","[64, 67, 52, 64, 61, 52, 57, 53, 52, 63, 67, 5...",44,196,64.60
8,family,single,"[943028, 261321, 858258, 831030, 1171925, 8535...","[26, 16, 15, 14, 40, 15, 21, 16, 14, 16, 13, 1...",12,40,16.93
9,family,multi,"[151899, 3782, 12994, 1437, 973362, 2428, 1512...","[130, 23, 20, 24, 31, 22, 119, 73, 18, 65, 64,...",17,141,47.72


In [133]:
# write val_test_og_df to parquet
val_test_og_df.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_val_test_og_summary.parquet', index=False)

In [105]:
# read /scratch/suyuelyu/deimm/data/oma/metadata_all_w_seq.parquet
all_meta_w_seq = pd.read_parquet('/scratch/suyuelyu/deimm/data/oma/metadata_all_w_seq.parquet')

In [ ]:
# get sequence for val test train
protein_to_seq = dict(zip(all_meta_w_seq['protein'], all_meta_w_seq['seq']))
del all_meta_w_seq

In [134]:
# map to sequence for val test train
meta_val_test['seq'] = meta_val_test['protein'].map(protein_to_seq)
meta_train_no_cluster_leak['seq'] = meta_train_no_cluster_leak['protein'].map(protein_to_seq)

/tmp/ipykernel_2570994/2127024044.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_val_test['seq'] = meta_val_test['protein'].map(protein_to_seq)
/tmp/ipykernel_2570994/2127024044.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_train_no_cluster_leak['seq'] = meta_train_no_cluster_leak['protein'].map(protein_to_seq)


In [135]:
# filter for length within 1024
meta_val_test_filtered = meta_val_test[meta_val_test['seq'].str.len() <= 1024]
meta_train_filtered = meta_train_no_cluster_leak[meta_train_no_cluster_leak['seq'].str.len() <= 1024]
print(f"Number of val test sequences after length filter: {len(meta_val_test_filtered)}, number of val test OGs after length filter: {meta_val_test_filtered['og'].nunique()}")
print(f"Number of train sequences after length filter: {len(meta_train_filtered)}, number of train OGs after length filter: {meta_train_filtered['og'].nunique()}")

Number of val test sequences after length filter: 269845, number of val test OGs after length filter: 1223
Number of train sequences after length filter: 12007816, number of train OGs after length filter: 654722


In [136]:
# save meta_val_test_filtered
meta_val_test_filtered.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_meta_val_test_l1024_all.parquet', index=False)
meta_train_filtered.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_meta_train_l1024_all.parquet', index=False)

In [137]:
# randomly assign half ogs to val and half to test in val_test_og_df
val_test_og_df['val_og'] = val_test_og_df.apply(lambda row: np.random.choice(row['ogs'], size=len(row['ogs'])//5, replace=False).tolist(), axis=1)
val_test_og_df['test_og'] = val_test_og_df.apply(lambda row: [og for og in row['ogs'] if og not in row['val_og']], axis=1)
val_test_og_df

,rank,group,ogs,og_size,min_og_size,max_og_size,mean_og_size,val_og,test_og
0,domain,single,"[770260, 930189, 1240134, 961038, 1238252, 122...","[493, 455, 502, 518, 1061, 107, 490, 1353, 480...",107,1353,572.30,"[1183523, 1134259, 674793, 1216640, 913903, 12...","[770260, 930189, 1240134, 961038, 1238252, 122..."
1,domain,multi,"[1248695, 921722, 1236080, 1217811, 711362, 12...","[2118, 1620, 1988, 1239, 1477, 1433, 1662, 158...",460,2254,1337.25,"[1204620, 1199762, 1248561, 1240084, 950127, 9...","[1248695, 921722, 1236080, 711362, 1223147, 12..."
2,phylum,single,"[1239313, 343149, 341795, 1244090, 1215031, 81...","[120, 142, 135, 135, 85, 42, 103, 109, 122, 14...",17,147,105.09,"[1142060, 1178849, 1156196, 1212585, 1196471, ...","[1239313, 343149, 341795, 1215031, 814260, 794..."
3,phylum,multi,"[699505, 1203908, 960958, 960947, 1235292, 691...","[524, 504, 515, 525, 476, 443, 535, 475, 567, ...",404,1489,544.40,"[1244069, 960951, 1196418, 1219605, 889979, 96...","[699505, 1203908, 960958, 960947, 691037, 7616..."
4,class,single,"[923689, 546708, 618759, 673331, 547234, 54583...","[43, 59, 46, 54, 58, 47, 52, 58, 59, 60, 55, 5...",40,66,55.09,"[1189116, 975502, 628097, 645559, 628588, 8811...","[923689, 618759, 673331, 547234, 545838, 62980..."
5,class,multi,"[349252, 1238424, 941963, 1156219, 967819, 115...","[136, 134, 133, 141, 133, 139, 139, 133, 133, ...",121,144,134.13,"[343203, 1151131, 1238424, 1185400, 1139804, 1...","[349252, 941963, 1156219, 967819, 1152861, 906..."
6,order,single,"[815076, 816970, 831377, 868566, 853980, 34230...","[12, 13, 18, 15, 16, 14, 15, 14, 13, 15, 18, 1...",10,136,18.49,"[810379, 982919, 1245727, 410425, 418095, 3571...","[816970, 831377, 868566, 853980, 342303, 34462..."
7,order,multi,"[1183778, 413092, 364521, 415668, 355402, 9799...","[64, 67, 52, 64, 61, 52, 57, 53, 52, 63, 67, 5...",44,196,64.60,"[1189139, 975682, 979945, 1136429, 1194107, 34...","[1183778, 364521, 415668, 355402, 912318, 9549..."
8,family,single,"[943028, 261321, 858258, 831030, 1171925, 8535...","[26, 16, 15, 14, 40, 15, 21, 16, 14, 16, 13, 1...",12,40,16.93,"[902280, 943028, 47969, 997547, 548086, 562561...","[261321, 858258, 831030, 1171925, 853549, 9430..."
9,family,multi,"[151899, 3782, 12994, 1437, 973362, 2428, 1512...","[130, 23, 20, 24, 31, 22, 119, 73, 18, 65, 64,...",17,141,47.72,"[3500, 630, 1136288, 1174092, 151292, 3838, 15...","[151899, 3782, 12994, 1437, 973362, 2428, 8306..."


In [138]:
meta_val = meta_val_test_filtered[meta_val_test_filtered['og'].isin(val_test_og_df['val_og'].explode())]
meta_test = meta_val_test_filtered[meta_val_test_filtered['og'].isin(val_test_og_df['test_og'].explode())]
print(f"Number of val sequences: {len(meta_val)}")
print(f"Number of val OGs: {meta_val['og'].nunique()}")
print(f"Number of test sequences: {len(meta_test)}")
print(f"Number of test OGs: {meta_test['og'].nunique()}")

Number of val sequences: 55982
Number of val OGs: 247
Number of test sequences: 216747
Number of test OGs: 986


In [140]:
# write meta_val and meta_test to parquet
meta_val.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_meta_val_l1024_all.parquet', index=False)
meta_test.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_meta_test_l1024_all.parquet', index=False)
val_test_og_df.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_val_test_og_summary.parquet', index=False)

In [141]:
og_summary_sorted_multi_species_train = og_summary_sorted_multi_species[og_summary_sorted_multi_species.index.isin(meta_train_no_cluster_leak['og'])]
og_summary_sorted_multi_species_train

,domain,phylum,class,order,family,genus,species,cluster_count,n_sequences
og,,,,,,,,,
1250885,3,82,182,378,654,1106,1747,198,2267
1146506,3,82,178,350,599,1015,1629,361,2138
953976,3,81,186,364,599,937,1347,133,1536
1238393,3,81,179,365,636,1060,1638,569,2065
1212410,3,81,175,376,661,1081,1730,299,2250
...,...,...,...,...,...,...,...,...,...
99965,1,1,1,1,1,1,2,1,3
99980,1,1,1,1,1,1,2,1,3
9999,1,1,1,1,1,1,2,2,2


In [144]:
og_summary_sorted_multi_species_train = og_summary_sorted_multi_species_train[
    og_summary_sorted_multi_species_train["n_sequences"] > 10
]


In [ ]:
for rank in std_ranks:
    # print n og with multi unique values for this rank
    n_og_multi_rank = (og_summary_sorted_multi_species_train[rank] > 1).sum()
    print(f"Number of train OGs with multiple unique {rank}: {n_og_multi_rank}")
    n_og_single_rank = (og_summary_sorted_multi_species_train[rank] == 1).sum()
    print(f"Number of train OGs with single unique {rank}: {n_og_single_rank}")

Number of train OGs with multiple unique domain: 22244
Number of train OGs with single unique domain: 186449
Number of train OGs with multiple unique phylum: 103727
Number of train OGs with single unique phylum: 104966
Number of train OGs with multiple unique class: 145475
Number of train OGs with single unique class: 63218
Number of train OGs with multiple unique order: 186979
Number of train OGs with single unique order: 21714
Number of train OGs with multiple unique family: 194590
Number of train OGs with single unique family: 14103
Number of train OGs with multiple unique genus: 201512
Number of train OGs with single unique genus: 7181
Number of train OGs with multiple unique species: 208693
Number of train OGs with single unique species: 0


In [ ]:
# 1 TB // 84 MB (1024 aa, all 32 hidden, assuming 4 bytes per float)
1000_000_000_000 // 84_000_000  # 11904 groups max

11904

In [163]:
# down sample to 12k groups at same composiiton of single vs multi rank groups
target_n_ogs = 12000
seed = 3515
og_summary_sorted_multi_species_train_12k = []
for i, rank in enumerate(std_ranks):
    # pick val test group with single R vs multi R
    og_single_rank = og_summary_sorted_multi_species_train[
        og_summary_sorted_multi_species_train[rank] == 1
    ]
    if i > 0:
        # need to make sure multi R is within same R-1 group
        prev_rank = std_ranks[i - 1]
        og_multi_rank = og_summary_sorted_multi_species_train[
            (og_summary_sorted_multi_species_train[rank] > 1)
            & (og_summary_sorted_multi_species_train[prev_rank] == 1)
        ]
    else:
        og_multi_rank = og_summary_sorted_multi_species_train[
            og_summary_sorted_multi_species_train[rank] > 1
        ]
    # rank both by n next rank
    if i < len(std_ranks) - 1:
        next_rank = std_ranks[i + 1]
        og_single_rank = og_single_rank.sort_values(by=next_rank, ascending=False)
        og_multi_rank = og_multi_rank.sort_values(by=next_rank, ascending=False)
    else:
        og_single_rank = og_single_rank.sort_values(by=rank, ascending=False)
        og_multi_rank = og_multi_rank.sort_values(by=rank, ascending=False)

    # take top 1000 from each group to sample from
    # og_single_rank = og_single_rank.head(500)
    # og_multi_rank = og_multi_rank.head(500)

    ratio_single = len(og_single_rank) / len(og_summary_sorted_multi_species_train)
    n_single_sample = int(ratio_single * target_n_ogs)
    ratio_multi = len(og_multi_rank) / len(og_summary_sorted_multi_species_train)
    n_multi_sample = int(ratio_multi * target_n_ogs)
    og_single_rank_sample = og_single_rank.sample(n=n_single_sample, random_state=seed).index.tolist()
    seed += 10
    og_multi_rank_sample = og_multi_rank.sample(n=n_multi_sample, random_state=seed).index.tolist()
    seed += 10
    og_summary_sorted_multi_species_train_12k.append((rank, 'single', og_single_rank_sample))
    og_summary_sorted_multi_species_train_12k.append((rank, 'multi', og_multi_rank_sample))
og_summary_sorted_multi_species_train_12k_df = pd.DataFrame(og_summary_sorted_multi_species_train_12k, columns=['rank', 'group', 'ogs'])

In [165]:
og_summary_sorted_multi_species_train_12k_df['n_ogs'] = og_summary_sorted_multi_species_train_12k_df['ogs'].apply(len)
og_summary_sorted_multi_species_train_12k_df

,rank,group,ogs,n_ogs
0,domain,single,"[736489, 1006143, 946840, 954763, 1145782, 124...",10720
1,domain,multi,"[910335, 943346, 807559, 662723, 728767, 11986...",1279
2,phylum,single,"[983443, 720450, 3852, 810452, 868426, 733186,...",6035
3,phylum,multi,"[794108, 334679, 866911, 986359, 995436, 82532...",4685
4,class,single,"[935688, 54394, 631611, 874895, 923647, 351912...",3635
5,class,multi,"[974568, 711007, 1169295, 258334, 341992, 1158...",2400
6,order,single,"[791713, 23980, 1162086, 986330, 452599, 82381...",1248
7,order,multi,"[1167884, 981953, 835633, 1149768, 799184, 931...",2386
8,family,single,"[658841, 160119, 28383, 849800, 8862, 1068196,...",810
9,family,multi,"[853641, 156931, 901995, 42582, 890998, 126803...",437


In [166]:
# filter meta_train_no_cluster_leak for ogs in og_summary_sorted_multi_species_train_12k_df
meta_train_no_cluster_leak_12k = meta_train_no_cluster_leak[meta_train_no_cluster_leak['og'].isin(og_summary_sorted_multi_species_train_12k_df['ogs'].explode())]
# meta_train_no_cluster_leak_12k print number of sequences and ogs
print(f"Number of train sequences in 12k set: {len(meta_train_no_cluster_leak_12k)}")
print(f"Number of train OGs in 12k set: {meta_train_no_cluster_leak_12k['og'].nunique()}")

Number of train sequences in 12k set: 1235438
Number of train OGs in 12k set: 32358


In [171]:
# check if there is any species that exist only in val test set but not in train set
# train_species = set(meta_train_no_cluster_leak_12k['taxid'].unique())
# val_test_species = set(meta_val_test_filtered['taxid'].unique())
# species_only_in_val_test = val_test_species - train_species
# print(f"Number of species only in val test set: {len(species_only_in_val_test)}")

for rank in std_ranks:
    uniq_rank_train = set(meta_train_no_cluster_leak_12k[rank].unique())
    uniq_rank_val_test = set(meta_val_test_filtered[rank].unique())
    rank_only_in_val_test = uniq_rank_val_test - uniq_rank_train
    print(f"Number of {rank} only in val test set: {len(rank_only_in_val_test)}")
    # are val test ranks well supported in train set?
    # quantify by min number of sequences in train set for each uniq_rank_val_test
    min_seq_per_rank = meta_train_no_cluster_leak_12k[meta_train_no_cluster_leak_12k[rank].isin(uniq_rank_val_test)].groupby(rank)['protein'].nunique().min()
    print(f"Minimum number of sequences in train set for {rank} in val test set: {min_seq_per_rank}")

Number of domain only in val test set: 0
Minimum number of sequences in train set for domain in val test set: 19832
Number of phylum only in val test set: 0
Minimum number of sequences in train set for phylum in val test set: 10
Number of class only in val test set: 0
Minimum number of sequences in train set for class in val test set: 10
Number of order only in val test set: 0
Minimum number of sequences in train set for order in val test set: 10
Number of family only in val test set: 0
Minimum number of sequences in train set for family in val test set: 10
Number of genus only in val test set: 0
Minimum number of sequences in train set for genus in val test set: 10
Number of species only in val test set: 0
Minimum number of sequences in train set for species in val test set: 6


In [172]:
# save 12k meta train set to parquet
meta_train_no_cluster_leak_12k.to_parquet('/scratch/suyuelyu/deimm/data/oma/oma_probe_meta_train_l1024_12k_ogs.parquet', index=False)

In [ ]:
# /scratch/suyuelyu/deimm/data/oma/oma_probe_last_protein_hidden_val_0.pkl
with open('/scratch/suyuelyu/deimm/data/oma/oma_probe_last_protein_hidden_val_0.pkl', 'rb') as f:
    last_hidden_val = pickle.load(f)


In [5]:
last_hidden_val[0]["last_protein_hiddens"][0].shape

torch.Size([389, 1280])

In [6]:
len(last_hidden_val[0]["last_protein_hiddens"])

33

In [7]:
last_hidden_val[0]["last_protein_hiddens"][0].dtype

torch.float16

In [9]:
last_hidden_val[0]

{'og': '554213',
 'protein_names': ['MAIZE34157',
  'ORYPU00709',
  'SORBI16252',
  'AEGTS19004',
  'ORYRU00807',
  'ORYNI28476',
  'WHEAT55835',
  'SETVI34409',
  'ERATE13167',
  'MISSI29983',
  'ORYSJ00812',
  'ORYBR00715',
  'BRADI08444',
  'ORYSI00859',
  'HORVV62017',
  'SETIT17338',
  'AEGTA27536'],
 'last_protein_hiddens': [tensor([[ 0.0080, -0.0031, -0.0404,  ..., -0.0147, -0.0217,  0.0263],
          [ 0.0006, -0.0098, -0.0077,  ...,  0.0022,  0.0083, -0.0013],
          [-0.0256, -0.0162,  0.0476,  ...,  0.0563, -0.0053, -0.0463],
          ...,
          [-0.0008, -0.0070, -0.0166,  ..., -0.0461,  0.0049, -0.0215],
          [ 0.0006, -0.0098, -0.0077,  ...,  0.0022,  0.0083, -0.0013],
          [-0.0090,  0.2162, -0.0299,  ...,  0.0763,  0.0130,  0.0197]],
         dtype=torch.float16),
  tensor([[-0.2029,  0.2517, -0.0665,  ..., -0.1859,  0.0068,  0.3181],
          [-0.3252, -0.3699,  0.1536,  ..., -0.1803,  0.0611,  0.4434],
          [ 0.1560, -0.0131,  0.0885,  ..., -0